In [21]:
import pandas as pd
from alive_progress import alive_bar
print("Finished imports.")
# import osmnx as ox

Finished imports.


In [22]:
# # Defining the map boundaries for Seattle Downtown Area
# north, south, east, west = 47.60171959708996, 47.595863783720105, -122.31146573381761, -122.32242746142184
# # Downloading the map as a graph object
# graph = ox.graph_from_bbox(bbox=(north, south, east, west), network_type = 'walk')
# # Retrieve nodes and edges geoDataFrame from MultiDiGraph graph
# nodes, edges = ox.graph_to_gdfs(graph)
# edges.to_csv('../data/edges_saigon.csv', index=True)
# edges.head()

In [23]:
edges_df = pd.read_csv('../data/edges_saigon.csv')

# type_severity_df = pd.read_csv(r'C:\Users\vzhang\OneDrive - Eastside Preparatory School\Documents\Walkability\access_type_severity.csv')
proj_sidewalk_df = pd.read_csv('../data/project_sidewalk_labels_saigon.csv')

type_severity_dict = {'Obstacle': 0.61,
                      'SurfaceProblem': 0.5,
                      'CurbRamp': 0.33,
                      'NoCurbRamp': 0.5,
                      'NoSidewalk': 0.5,
                      'Occlusion': 0.5,
                      'Other': 0.5}

print("Finished loading dataframes.")


Finished loading dataframes.


In [24]:
def make_note(labels_sev: list) -> list:
    """
    Aggregates severity values for each label and returns summary statistics.

    Parameters:
    -----------
    labels_sev : list of dict
        A list where each element is a dictionary with a single key-value pair,
        representing a label and its corresponding severity score (numeric).

    Returns:
    --------
    list of dict
        A list of dictionaries where each key is a label, and the value is a list
        containing:
            - The count of how many times the label appeared.
            - The average severity for that label.
    """
    final = {}
    for pair in labels_sev:
        for label, severity in pair.items():
            if label in final:
                final[label].append(severity)
            else:
                final[label] = [severity]

    messages = []
    for label in final:
        avg_sev = sum(final[label]) / len(final[label])
        messages.append({label: [len(final[label]), avg_sev]})

    return messages


In [25]:
access_scores = []  # List to store computed accessibility scores per edge
notes = []          # List to store annotation notes per edge

# Progress bar for iterating over edges_df
with alive_bar(len(edges_df), force_tty=True) as bar:
    try:
        for index, row in edges_df.iterrows():
            osm_id = row['osmid']  # Get the OSM ID for the current street segment
            same_segment = []       # List of weighted severity values for this segment
            segment_labels = []     # List of label/severity dicts for this segment

            # Iterate through all sidewalk labels to find those on the same segment
            for _, r in proj_sidewalk_df.iterrows():
                if r['OSM Street ID'] == osm_id:
                    if r['Label Type'] in type_severity_dict:
                        # Compute weighted severity: (severity/5) * weight
                        severity = r['Severity']
                        weight = type_severity_dict[r['Label Type']]
                        same_segment.append(severity / 5 * weight)

                        # Save label info for later notes
                        segment_labels.append({r['Label Type']: severity})

            # Compute accessibility score:
            # Formula: 1 - min(1, total_weighted_severity * (density factor))
            score = 1 - min(1, (sum(same_segment) * (len(same_segment) / min(row['length'], 150))))
            access_scores.append(score)

            # If there are any labels, process them into summarized notes
            if segment_labels:
                notes.append(make_note(segment_labels))
            else:
                notes.append({})  # No labels found for this segment

            bar()  # Update progress bar

    except Exception as e:
        # If an error occurs, print partial results and re-raise the error
        print("Partial access_scores:", access_scores)
        raise e

print("Finished Assigning Scores.")

|████████████████████████████████████████| 2456/2456 [100%] in 1:31.8 (26.76/s)  ▅▇▇ 336/2456 [14%] in 14s (~1:30, 23. ▇▇▅ 1032/2456 [42%] in 40s (~56s, 25.
Finished Assigning Scores.


In [26]:
edges_df['AccessScore'] = access_scores # save the new columns
edges_df["sidewalk_notes"] = notes
edges_df.to_csv("../output/edges_saigon_scored.csv", index=False)